# 02 - Feature Engineering

Ce notebook prépare les données brutes pour l'entraînement des modèles de recommandation.
Il charge les jeux de données MovieLens 100K, MovieLens 1M et RetailRocket, nettoie les interactions, encode les IDs, et extrait les features contextuelles.

L'objectif est de produire des versions prêtes à l'emploi dans `data/processed/`.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str((Path('..') / 'src').resolve()))
from data_utils import load_movielens_100k, load_movielens_1m, load_retailrocket, filter_interactions, encode_ids
from context_features import extract_all_context_features

print('Python:', sys.version.split()[0])
print('pandas :', pd.__version__)
print('numpy :', np.__version__)


## Chemins des données

On définit les chemins d'accès pour les données brutes et les données traitées.


In [ ]:
NOTEBOOK_ROOT = Path('.')
RAW_ROOT = (NOTEBOOK_ROOT / '..' / 'data' / 'raw').resolve()
PROCESSED_ROOT = (NOTEBOOK_ROOT / '..' / 'data' / 'processed').resolve()
print('RAW_ROOT =', RAW_ROOT)
print('PROCESSED_ROOT =', PROCESSED_ROOT)

processed_paths = {
    'ml100k': PROCESSED_ROOT / 'movielens_100k',
    'ml1m': PROCESSED_ROOT / 'movielens_1m',
    'retailrocket': PROCESSED_ROOT / 'retailrocket'
}
for path in processed_paths.values():
    path.mkdir(parents=True, exist_ok=True)
    print(path)


## 1. MovieLens 100K

Nous nettoyons et enrichissons le dataset MovieLens 100K.
Nous appliquons un filtrage itératif pour supprimer les utilisateurs et items avec moins de 5 interactions, puis nous extrayons des features temporelles et de session.


In [ ]:
ml100k_dir = RAW_ROOT / 'movielens' / 'ml-100k'
ml100k = load_movielens_100k(ml100k_dir)
print('raw ml100k shape:', ml100k.shape)
print(ml100k.head())

ml100k_filtered = filter_interactions(ml100k, min_interactions=5)
print('filtered ml100k shape:', ml100k_filtered.shape)

ml100k_encoded = encode_ids(ml100k_filtered)
ml100k_features = extract_all_context_features(ml100k_encoded)

print('processed ml100k columns:', ml100k_features.columns.tolist())
print(ml100k_features[['user_id_encoded', 'item_id_encoded', 'hour_of_day', 'dow_sin', 'session_length']].head())


In [ ]:
ml100k_features.to_parquet(processed_paths['ml100k'] / 'movielens_100k_processed.parquet', index=False)
print('Saved MovieLens 100K processed dataset.')


## 2. MovieLens 1M

Le même pipeline s'applique à MovieLens 1M pour obtenir un deuxième dataset d'entraînement.


In [ ]:
ml1m_dir = RAW_ROOT / 'movielens' / 'ml-1m'
ml1m = load_movielens_1m(ml1m_dir)
print('raw ml1m shape:', ml1m.shape)
print(ml1m.head())

ml1m_filtered = filter_interactions(ml1m, min_interactions=5)
print('filtered ml1m shape:', ml1m_filtered.shape)

ml1m_encoded = encode_ids(ml1m_filtered)
ml1m_features = extract_all_context_features(ml1m_encoded)

print('processed ml1m columns:', ml1m_features.columns.tolist())
print(ml1m_features[['user_id_encoded', 'item_id_encoded', 'hour_of_day', 'dow_cos', 'time_since_last_interaction_log']].head())


In [ ]:
ml1m_features.to_parquet(processed_paths['ml1m'] / 'movielens_1m_processed.parquet', index=False)
print('Saved MovieLens 1M processed dataset.')


## 3. RetailRocket

RetailRocket utilise des événements implicites (`view`, `addtocart`, `transaction`).
Nous codons ces types d'événements et extrayons les mêmes features de session et temporelles.


In [ ]:
rr = load_retailrocket(RAW_ROOT / 'retailrocket')
print('raw retailrocket shape:', rr.shape)
print(rr.head())

event_map = {'view': 0, 'addtocart': 1, 'transaction': 2}
rr['event_type'] = rr['event'].map(event_map).fillna(-1).astype(int)
rr['event_weight'] = rr['event_type'] + 1

rr_filtered = filter_interactions(rr, min_interactions=5)
print('filtered retailrocket shape:', rr_filtered.shape)

rr_encoded = encode_ids(rr_filtered)
rr_features = extract_all_context_features(rr_encoded, is_retail_rocket=True)

print('processed retailrocket columns:', rr_features.columns.tolist())
print(rr_features[['user_id_encoded', 'item_id_encoded', 'event_type', 'session_length', 'is_desktop_proxy']].head())


In [ ]:
rr_features.to_parquet(processed_paths['retailrocket'] / 'retailrocket_processed.parquet', index=False)
print('Saved RetailRocket processed dataset.')


## Vérification finale

On vérifie que les fichiers de sortie sont bien présents et que les colonnes clés sont disponibles.


In [ ]:
for name, path in processed_paths.items():
    files = sorted(path.glob('*.parquet'))
    print(name, '->', files)


## Notes

- Les datasets traités sont stockés dans `data/processed/` pour être réutilisés par les notebooks d'entraînement.
- Les IDs utilisateur et item sont encodés en entiers contigus (`user_id_encoded`, `item_id_encoded`).
- Les features temporelles et de session sont extraites pour enrichir les modèles contextuels.
